In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [6]:
# 1)
import os
import torch
import transformers
import datasets
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
PEFT: 0.21.2
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [7]:
# 2)
import pandas as pd
import numpy as np
import json

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [8]:
# 3)
engsaf = load_dataset("IsmaelMousa/engsaf")

print("DATASET LOADED SUCCESSFULLY")
print(engsaf)

README.md:   0%|          | 0.00/13.3k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/4.07M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/290k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/432k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4735 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/338 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/506 [00:00<?, ? examples/s]

DATASET LOADED SUCCESSFULLY
DatasetDict({
    train: Dataset({
        features: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale'],
        num_rows: 4735
    })
    validation: Dataset({
        features: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale'],
        num_rows: 338
    })
    test: Dataset({
        features: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale'],
        num_rows: 506
    })
})


In [10]:
# 4)
for split in engsaf:
    print(f"\n{split.upper()}: {len(engsaf[split])} rows")
    print("Columns:", engsaf[split].column_names)

print("\nFIRST TRAINING EXAMPLE:")
print(engsaf["train"][0])


TRAIN: 4735 rows
Columns: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale']

VALIDATION: 338 rows
Columns: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale']

TEST: 506 rows
Columns: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale']

FIRST TRAINING EXAMPLE:
{'question': 'Q1. State TRUE or FALSE and justify. No correct justification, no marks.\ne. If a parent process forks a new child process, the exec system call cannot be used in the parent process until the child process terminates.', 'student_answer': 'FALSE', 'reference_answer': 'False, parent and child processes are two independent processes. Parent can perform any operation that it likes including an exec system call.', 'mark_scheme': "{'0': 'Incorrect response', '1': 'Partially correct response', '2': 'Correct response'}", 'score': 0, 'rationale': 'Your answer is incorrect. Parent and child processes are independent 

In [11]:
# 5)
# STEP 5: Clean empty student answers

train_clean = pd.DataFrame(engsaf["train"]).copy()
val_clean = pd.DataFrame(engsaf["validation"]).copy()
test_clean = pd.DataFrame(engsaf["test"]).copy()

def clean_answers(df):
    return df[
        df["student_answer"].fillna("").astype(str).str.strip().ne("")
    ].reset_index(drop=True)

train_clean = clean_answers(train_clean)
val_clean = clean_answers(val_clean)
test_clean = clean_answers(test_clean)

print("TRAIN:", len(train_clean))
print("VALIDATION:", len(val_clean))
print("TEST:", len(test_clean))

TRAIN: 4734
VALIDATION: 338
TEST: 506


In [28]:
# 6)
# STEP 6: Rebuild leakage-free splits by question

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Combine the existing cleaned datasets
all_data = pd.concat(
    [train_clean, val_clean, test_clean],
    ignore_index=True
)

# Normalize question text for grouping
all_data["_question_group"] = (
    all_data["question"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# First split: 80% train, 20% temporary
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(all_data, groups=all_data["_question_group"])
)

train_clean = all_data.iloc[train_idx].copy()
temp_data = all_data.iloc[temp_idx].copy()

# Second split: divide remaining 20% equally
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    split2.split(temp_data, groups=temp_data["_question_group"])
)

val_clean = temp_data.iloc[val_idx].copy()
test_clean = temp_data.iloc[test_idx].copy()

# Remove temporary grouping column
for df in [train_clean, val_clean, test_clean]:
    df.drop(columns=["_question_group"], inplace=True)

# Verify leakage
train_q = set(train_clean["question"].str.strip().str.lower())
val_q = set(val_clean["question"].str.strip().str.lower())
test_q = set(test_clean["question"].str.strip().str.lower())

print("TRAIN:", len(train_clean))
print("VALIDATION:", len(val_clean))
print("TEST:", len(test_clean))

print("\nTrain ∩ Validation:", len(train_q & val_q))
print("Train ∩ Test:", len(train_q & test_q))
print("Validation ∩ Test:", len(val_q & test_q))


TRAIN: 4403
VALIDATION: 533
TEST: 642

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [30]:
# 7)
# STEP 7: Define Kaizenly system prompt

SYSTEM_PROMPT = """You are Kaizenly, an academic assignment feedback assistant.
Evaluate the student's answer using the question, reference answer,
and marking scheme.

Return valid JSON with these fields:
- score: integer (0, 1, or 2)
- performance: Incorrect, Partially Correct, or Correct
- feedback: explanation of the evaluation

Use the provided reference answer and rationale.
Do not invent information or award unsupported marks."""

In [33]:
# 8)
# STEP 8: Format one example

import json

def format_example_v2(row):

    user_prompt = f"""Question:
{row['question']}

Student Answer:
{row['student_answer']}

Reference Answer:
{row['reference_answer']}

Mark Scheme:
{row['mark_scheme']}"""

    score = int(row["score"])

    performance = {
        0: "Incorrect",
        1: "Partially Correct",
        2: "Correct"
    }[score]

    assistant_response = {
        "score": score,
        "performance": performance,
        "feedback": row["rationale"]
    }

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {
                "role": "assistant",
                "content": json.dumps(assistant_response, ensure_ascii=False)
            }
        ]
    }

In [35]:
# 9)
# STEP 9: Format corrected datasets

train_formatted = [
    format_example_v2(row)
    for _, row in train_clean.iterrows()
]

val_formatted = [
    format_example_v2(row)
    for _, row in val_clean.iterrows()
]

test_formatted = [
    format_example_v2(row)
    for _, row in test_clean.iterrows()
]

print("Training:", len(train_formatted))
print("Validation:", len(val_formatted))
print("Testing:", len(test_formatted))


Training: 4403
Validation: 533
Testing: 642


In [37]:
# 10)
# STEP 10: Inspect formatted example

sample = train_formatted[0]

print("SAMPLE TYPE:", type(sample))
print("MESSAGE COUNT:", len(sample["messages"]))

for message in sample["messages"]:
    print("\nROLE:", message["role"])
    print("CONTENT:\n", message["content"][:1000])

SAMPLE TYPE: <class 'dict'>
MESSAGE COUNT: 3

ROLE: system
CONTENT:
 You are Kaizenly, an academic assignment feedback assistant.
Evaluate the student's answer using the question, reference answer,
and marking scheme.

Return valid JSON with these fields:
- score: integer (0, 1, or 2)
- performance: Incorrect, Partially Correct, or Correct
- feedback: explanation of the evaluation

Use the provided reference answer and rationale.
Do not invent information or award unsupported marks.

ROLE: user
CONTENT:
 Question:
Q1. State TRUE or FALSE and justify. No correct justification, no marks.
e. If a parent process forks a new child process, the exec system call cannot be used in the parent process until the child process terminates.

Student Answer:
FALSE

Reference Answer:
False, parent and child processes are two independent processes. Parent can perform any operation that it likes including an exec system call.

Mark Scheme:
{'0': 'Incorrect response', '1': 'Partially correct response', '

In [38]:
# STEP 11: Tokenization check

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text_to_test = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=False,
    add_generation_prompt=False
)

tokens = tokenizer(
    text_to_test,
    add_special_tokens=False
)["input_ids"]

print("Tokenizer:", MODEL_NAME)
print("Formatted text length:", len(text_to_test))
print("Token count:", len(tokens))
print("\nFIRST 500 CHARACTERS:\n", text_to_test[:500])

Tokenizer: Qwen/Qwen2.5-1.5B-Instruct
Formatted text length: 1286
Token count: 261

FIRST 500 CHARACTERS:
 <|im_start|>system
You are Kaizenly, an academic assignment feedback assistant.
Evaluate the student's answer using the question, reference answer,
and marking scheme.

Return valid JSON with these fields:
- score: integer (0, 1, or 2)
- performance: Incorrect, Partially Correct, or Correct
- feedback: explanation of the evaluation

Use the provided reference answer and rationale.
Do not invent information or award unsupported marks.<|im_end|>
<|im_start|>user
Question:
Q1. State TRUE or FALSE a


In [39]:
# STEP 12: Score distribution

print("TRAIN SCORES")
print(train_clean["score"].value_counts().sort_index())

print("\nVALIDATION SCORES")
print(val_clean["score"].value_counts().sort_index())

print("\nTEST SCORES")
print(test_clean["score"].value_counts().sort_index())

TRAIN SCORES
score
0     953
1    1604
2    1846
Name: count, dtype: int64

VALIDATION SCORES
score
0    109
1    192
2    232
Name: count, dtype: int64

TEST SCORES
score
0    138
1    197
2    307
Name: count, dtype: int64


In [40]:
# STEP 13: Missing value check

columns = [
    "question",
    "student_answer",
    "reference_answer",
    "mark_scheme",
    "score",
    "rationale"
]

for name, df in [
    ("TRAIN", train_clean),
    ("VALIDATION", val_clean),
    ("TEST", test_clean)
]:
    print(f"\n{name}")
    print(df[columns].isnull().sum())


TRAIN
question            0
student_answer      0
reference_answer    0
mark_scheme         0
score               0
rationale           0
dtype: int64

VALIDATION
question            0
student_answer      0
reference_answer    0
mark_scheme         0
score               0
rationale           0
dtype: int64

TEST
question            0
student_answer      0
reference_answer    0
mark_scheme         0
score               0
rationale           0
dtype: int64


In [41]:
# STEP 14: Validate score labels

for name, df in [
    ("TRAIN", train_clean),
    ("VALIDATION", val_clean),
    ("TEST", test_clean)
]:
    invalid = df[~df["score"].isin([0, 1, 2])]

    print(name, "Invalid scores:", len(invalid))

TRAIN Invalid scores: 0
VALIDATION Invalid scores: 0
TEST Invalid scores: 0


In [42]:
# STEP 15: Token length analysis

def get_token_length(example):
    formatted = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return len(tokenizer(
        formatted,
        add_special_tokens=False
    )["input_ids"])


train_lengths = [get_token_length(x) for x in train_formatted]
val_lengths = [get_token_length(x) for x in val_formatted]
test_lengths = [get_token_length(x) for x in test_formatted]

import numpy as np

for name, lengths in [
    ("TRAIN", train_lengths),
    ("VALIDATION", val_lengths),
    ("TEST", test_lengths)
]:
    print(f"\n{name}")
    print("Average:", round(np.mean(lengths), 2))
    print("Maximum:", max(lengths))
    print("95th percentile:", int(np.percentile(lengths, 95)))


TRAIN
Average: 306.36
Maximum: 725
95th percentile: 458

VALIDATION
Average: 292.7
Maximum: 578
95th percentile: 391

TEST
Average: 308.4
Maximum: 457
95th percentile: 377


In [43]:
# STEP 16: Duplicate check

for name, df in [
    ("TRAIN", train_clean),
    ("VALIDATION", val_clean),
    ("TEST", test_clean)
]:
    duplicates = df.duplicated(
        subset=["question", "student_answer"]
    ).sum()

    print(name, "Duplicate examples:", duplicates)

TRAIN Duplicate examples: 14
VALIDATION Duplicate examples: 0
TEST Duplicate examples: 1


In [44]:

# STEP 17: Normalize questions and answers

def normalize_text(value):
    return " ".join(str(value).lower().split())

train_prepared = train_clean.copy()
val_prepared = val_clean.copy()
test_prepared = test_clean.copy()

for df in [train_prepared, val_prepared, test_prepared]:
    df["_question_key"] = df["question"].apply(normalize_text)
    df["_answer_key"] = df["student_answer"].apply(normalize_text)

print("Question normalization completed!")

print("Train:", len(train_prepared))
print("Validation:", len(val_prepared))
print("Test:", len(test_prepared))


Question normalization completed!
Train: 4403
Validation: 533
Test: 642


In [45]:

# STEP 18: Remove duplicates within each split

def remove_duplicates(df):
    return df.drop_duplicates(
        subset=["_question_key", "_answer_key"],
        keep="first"
    ).copy()

train_prepared = remove_duplicates(train_prepared)
val_prepared = remove_duplicates(val_prepared)
test_prepared = remove_duplicates(test_prepared)

print("AFTER DUPLICATE REMOVAL")
print("Train:", len(train_prepared))
print("Validation:", len(val_prepared))
print("Test:", len(test_prepared))


AFTER DUPLICATE REMOVAL
Train: 4325
Validation: 520
Test: 634


In [ ]:

# STEP 19: Create question-disjoint datasets

test_final = test_prepared.copy()

test_questions = set(test_final["_question_key"])

val_final = val_prepared[
    ~val_prepared["_question_key"].isin(test_questions)
].copy()

validation_questions = set(val_final["_question_key"])

train_final = train_prepared[
    ~train_prepared["_question_key"].isin(
        test_questions | validation_questions
    )
].copy()

print("FINAL DATASET SIZES")
print("Training:", len(train_final))
print("Validation:", len(val_final))
print("Testing:", len(test_final))

assert all(
    len(df) > 0
    for df in [train_final, val_final, test_final]
), "One of the datasets became empty!"


In [ ]:

# STEP 20: Verify data separation

train_questions = set(train_final["_question_key"])
val_questions = set(val_final["_question_key"])
test_questions = set(test_final["_question_key"])

print(
    "Train / Validation overlap:",
    len(train_questions & val_questions)
)

print(
    "Train / Test overlap:",
    len(train_questions & test_questions)
)

print(
    "Validation / Test overlap:",
    len(val_questions & test_questions)
)

assert not (train_questions & val_questions)
assert not (train_questions & test_questions)
assert not (val_questions & test_questions)

print("\nSUCCESS: No question overlap between splits!")


In [ ]:

# STEP 21: Rebuild Hugging Face datasets

from datasets import Dataset

train_formatted_final = [
    format_example_v2(row)
    for _, row in train_final.iterrows()
]

val_formatted_final = [
    format_example_v2(row)
    for _, row in val_final.iterrows()
]

test_formatted_final = [
    format_example_v2(row)
    for _, row in test_final.iterrows()
]

train_dataset = Dataset.from_list(train_formatted_final)
val_dataset = Dataset.from_list(val_formatted_final)
test_dataset = Dataset.from_list(test_formatted_final)

print("HUGGING FACE DATASETS READY!")

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

print("\nFeatures:", train_dataset.column_names)
print("Messages in first example:",
      len(train_dataset[0]["messages"]))


In [46]:

# STEP 22: Prepare development data

test_final = test_prepared.copy()

test_questions = set(test_final["_question_key"])

development_data = pd.concat(
    [train_prepared, val_prepared],
    ignore_index=True
)

development_data = development_data[
    ~development_data["_question_key"].isin(test_questions)
].copy()

print("Development examples:", len(development_data))
print(
    "Unique development questions:",
    development_data["_question_key"].nunique()
)
print("Test examples:", len(test_final))


Development examples: 4845
Unique development questions: 106
Test examples: 634


In [47]:

# STEP 23: Split by question groups

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        development_data,
        groups=development_data["_question_key"]
    )
)

train_final = development_data.iloc[train_idx].copy()
val_final = development_data.iloc[val_idx].copy()

print("Training:", len(train_final))
print("Validation:", len(val_final))
print("Testing:", len(test_final))


Training: 4035
Validation: 810
Testing: 634


In [48]:

# STEP 24: Final data quality verification

train_questions = set(train_final["_question_key"])
val_questions = set(val_final["_question_key"])
test_questions = set(test_final["_question_key"])

print("Train / Validation overlap:",
      len(train_questions & val_questions))

print("Train / Test overlap:",
      len(train_questions & test_questions))

print("Validation / Test overlap:",
      len(val_questions & test_questions))

print("\nSCORE DISTRIBUTION")

for name, df in [
    ("TRAIN", train_final),
    ("VALIDATION", val_final),
    ("TEST", test_final)
]:
    print(f"\n{name}:")
    print(df["score"].value_counts().sort_index())

assert not (train_questions & val_questions)
assert not (train_questions & test_questions)
assert not (val_questions & test_questions)

print("\nALL CHECKS PASSED!")


Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

SCORE DISTRIBUTION

TRAIN:
score
0     853
1    1438
2    1744
Name: count, dtype: int64

VALIDATION:
score
0    192
1    337
2    281
Name: count, dtype: int64

TEST:
score
0    137
1    192
2    305
Name: count, dtype: int64

ALL CHECKS PASSED!


In [49]:

# STEP 25: Format final datasets

train_formatted_final = [
    format_example_v2(row)
    for _, row in train_final.iterrows()
]

val_formatted_final = [
    format_example_v2(row)
    for _, row in val_final.iterrows()
]

test_formatted_final = [
    format_example_v2(row)
    for _, row in test_final.iterrows()
]

train_dataset = Dataset.from_list(train_formatted_final)
val_dataset = Dataset.from_list(val_formatted_final)
test_dataset = Dataset.from_list(test_formatted_final)

print("FINAL DATASETS READY!")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))


FINAL DATASETS READY!
Training: 4035
Validation: 810
Testing: 634


In [50]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("GPU not detected. Enable GPU in Kaggle Settings.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [51]:
# 1]
import torch
import transformers
import peft

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
PEFT: 0.21.2
GPU: Tesla T4


In [52]:
# 2]
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded successfully!")
print("Vocabulary size:", len(tokenizer))
print("Padding token:", tokenizer.pad_token)

Tokenizer loaded successfully!
Vocabulary size: 151665
Padding token: <|endoftext|>


In [53]:
# 3]
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configured!")

4-bit quantization configured!


In [54]:
%pip install -q -U bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [55]:
# 4]
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("Base model loaded successfully!")

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully!
GPU memory allocated: 0.96 GB


In [56]:
#5]
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [59]:

# # 6] Assistant-only tokenization (PyArrow compatible)

def tokenize_batch(batch):
    all_input_ids = []
    all_labels = []
    all_attention_masks = []

    for messages in batch["messages"]:

        # Full conversation as plain text
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        # System + user prompt, including assistant generation header
        prompt_text = tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=False,
            add_generation_prompt=True
        )

        # Convert text into ordinary Python lists of token IDs
        full_ids = tokenizer(
            full_text,
            add_special_tokens=False
        )["input_ids"]

        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False
        )["input_ids"]

        # Verify prompt is a prefix of the full conversation
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError("Prompt/full conversation token mismatch!")

        prompt_length = len(prompt_ids)

        # Ignore prompt tokens during training
        labels = [-100] * prompt_length + full_ids[prompt_length:]

        # Truncate consistently
        full_ids = full_ids[:1024]
        labels = labels[:1024]

        all_input_ids.append(full_ids)
        all_labels.append(labels)
        all_attention_masks.append([1] * len(full_ids))

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": all_attention_masks
    }


train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_dataset.column_names
)

print("Assistant-only tokenization completed!")
print("Training samples:", len(train_tokenized))
print("Validation samples:", len(val_tokenized))


Map:   0%|          | 0/4035 [00:00<?, ? examples/s]

Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Assistant-only tokenization completed!
Training samples: 4035
Validation samples: 810


In [68]:

# # 7] Custom assistant-only data collator
class AssistantOnlyCollator:
    def __call__(self, features):
        lengths = [len(f["input_ids"]) for f in features]
        max_len = max(lengths)

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f["input_ids"], dtype=torch.long) for f in features],
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        )

        labels = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f["labels"], dtype=torch.long) for f in features],
            batch_first=True,
            padding_value=-100
        )

        attention_mask = torch.zeros(
            (len(features), max_len), dtype=torch.long
        )

        for i, length in enumerate(lengths):
            attention_mask[i, :length] = 1

        return {
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": attention_mask
        }

data_collator = AssistantOnlyCollator()
print("Corrected collator ready!")

Corrected collator ready!


In [69]:
batch = data_collator([train_tokenized[i] for i in range(2)])

print("Input shape:", batch["input_ids"].shape)
print("Mask shape:", batch["attention_mask"].shape)
print("Supervised tokens:", (batch["labels"] != -100).sum().item())
print("Attention tokens:", batch["attention_mask"].sum().item())

# Confirm padding is masked from attention
print("Padding attention check:",
      batch["attention_mask"][0, len(train_tokenized[0]["input_ids"]):].sum().item())

Input shape: torch.Size([2, 261])
Mask shape: torch.Size([2, 261])
Supervised tokens: 117
Attention tokens: 502
Padding attention check: 0


In [70]:
# Final dataset and GPU checks

print("Train examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

print("Trainer model:", type(trainer.model).__name__)
print("Training epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Batch size:", training_args.per_device_train_batch_size)

Train examples: 4035
Validation examples: 810
GPU: Tesla T4
GPU memory allocated: 1.42 GB
GPU memory reserved: 1.9 GB


NameError: name 'trainer' is not defined

In [64]:

# # 8] Training configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./kaizenly-v2",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    num_train_epochs=2,
    learning_rate=5e-5,
    warmup_steps=50,
    weight_decay=0.01,

    fp16=True,

    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    optim="adamw_torch",
    report_to="none",
    gradient_checkpointing=True
)

print("Training configuration ready!")


Training configuration ready!


In [63]:

import transformers
import inspect
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)
print("\nSupported TrainingArguments:")
print(inspect.signature(TrainingArguments.__init__))


Transformers version: 5.20.2

Supported TrainingArguments:
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False, gradient_checkpointing_kwargs: dict[str, typing.Any]

In [67]:
# Verify assistant-only loss masking

sample = train_tokenized[0]

masked = sum(x == -100 for x in sample["labels"])
active = sum(x != -100 for x in sample["labels"])

print("Prompt tokens masked:", masked)
print("Assistant tokens trained:", active)
print("Total tokens:", len(sample["labels"]))

assert active > 0, "No assistant tokens available for training!"
assert masked > 0, "Prompt tokens were not masked!"

print("Assistant-only masking verified!")

Prompt tokens masked: 206
Assistant tokens trained: 55
Total tokens: 261
Assistant-only masking verified!


In [71]:
# 9]
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    data_collator=data_collator
)

print("Kaizenly trainer initialized!")

Kaizenly trainer initialized!


In [72]:
print("Trainer initialized:", trainer is not None)
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Model:", type(trainer.model).__name__)

Trainer initialized: True
Training examples: 4035
Validation examples: 810
Model: PeftModelForCausalLM


In [73]:
# Check supervised tokens across the complete datasets

train_empty = sum(
    1 for x in train_tokenized
    if not any(t != -100 for t in x["labels"])
)

val_empty = sum(
    1 for x in val_tokenized
    if not any(t != -100 for t in x["labels"])
)

print("Train examples without target tokens:", train_empty)
print("Validation examples without target tokens:", val_empty)

assert train_empty == 0
assert val_empty == 0

print("Dataset verification passed!")

Train examples without target tokens: 0
Validation examples without target tokens: 0
Dataset verification passed!


In [74]:
# 10] MAIN TRAINING
training_result = trainer.train()

print("TRAINING COMPLETED!")
print(training_result)

Step,Training Loss,Validation Loss
100,0.899562,0.964871
200,0.905312,0.932153
300,0.888880,0.911379
400,0.853540,0.901210
500,0.836929,0.895397
600,0.824549,0.895451
700,0.793919,0.890013
800,0.768959,0.886612
900,0.803166,0.883459
1000,0.785080,0.882814


TRAINING COMPLETED!
TrainOutput(global_step=1010, training_loss=0.8489065812365844, metrics={'train_runtime': 4583.0907, 'train_samples_per_second': 1.761, 'train_steps_per_second': 0.22, 'total_flos': 2.2302377608974336e+16, 'train_loss': 0.8489065812365844, 'epoch': 2.0})


In [76]:
# 11] Save the fine-tuned model
OUTPUT_DIR = "/kaggle/working/kaizenly-lora-adapter"

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Kaizenly fine-tuned adapter saved!")
print("Location:", OUTPUT_DIR)

Kaizenly fine-tuned adapter saved!
Location: /kaggle/working/kaizenly-lora-adapter


In [78]:
# 12] Create a downloadable ZIP
import shutil
import os

ZIP_PATH = "/kaggle/working/kaizenly-lora-adapter"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    OUTPUT_DIR
)

print("ZIP created successfully!")
print(os.path.getsize(ZIP_PATH + ".zip") / (1024 * 1024), "MB")

ZIP created successfully!
34.707621574401855 MB


In [79]:
# 13] Verify adapter integrity
import os
import json
import hashlib

adapter_path = "/kaggle/working/kaizenly-lora-adapter"

print("Folder exists:", os.path.exists(adapter_path))

with open(os.path.join(adapter_path, "adapter_config.json")) as f:
    config = json.load(f)

print("Base model:", config["base_model_name_or_path"])
print("LoRA rank:", config["r"])
print("LoRA alpha:", config["lora_alpha"])

weights = os.path.join(adapter_path, "adapter_model.safetensors")
print("Weights exist:", os.path.exists(weights))
print("Weights size (MB):", round(os.path.getsize(weights) / (1024 * 1024), 2))

with open(weights, "rb") as f:
    print("SHA256:", hashlib.sha256(f.read()).hexdigest())

Folder exists: True
Base model: Qwen/Qwen2.5-1.5B-Instruct
LoRA rank: 8
LoRA alpha: 16
Weights exist: True
Weights size (MB): 35.27
SHA256: 9e07a0aad6d9173613afa60b8f20fae276d2f58cdaf0ef0875320c0346dcd250


In [80]:
import os
import hashlib

zip_path = "/kaggle/working/kaizenly-lora-adapter.zip"

print("ZIP exists:", os.path.exists(zip_path))

if os.path.exists(zip_path):
    print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))
    with open(zip_path, "rb") as f:
        print("ZIP SHA256:", hashlib.sha256(f.read()).hexdigest())
else:
    print("ZIP not found. We can create it from the adapter folder.")

ZIP exists: True
ZIP size (MB): 34.71
ZIP SHA256: a627b0fff92740f555511bcc00bf9f3329b9d93f401e0da455af4c2078af7256


In [81]:
import os
import shutil
import hashlib

adapter_path = "/kaggle/working/kaizenly-lora-adapter"

# Create a fresh archive with a different name
zip_base = "/kaggle/working/kaizenly-lora-adapter-verified"

if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")

shutil.make_archive(zip_base, "zip", adapter_path)

# Verify the actual weights
weights_path = os.path.join(adapter_path, "adapter_model.safetensors")

with open(weights_path, "rb") as f:
    print("CURRENT WEIGHTS HASH:", hashlib.sha256(f.read()).hexdigest())

# Verify the new ZIP
with open(zip_base + ".zip", "rb") as f:
    print("FRESH ZIP HASH:", hashlib.sha256(f.read()).hexdigest())

print("FRESH ZIP SIZE (MB):", round(os.path.getsize(zip_base + ".zip") / (1024 * 1024), 2))

CURRENT WEIGHTS HASH: 9e07a0aad6d9173613afa60b8f20fae276d2f58cdaf0ef0875320c0346dcd250
FRESH ZIP HASH: a627b0fff92740f555511bcc00bf9f3329b9d93f401e0da455af4c2078af7256
FRESH ZIP SIZE (MB): 34.71


In [82]:
# 1} Locate the test dataset
import os

base = "/kaggle/working/kaizenly-training"

print("Training folder exists:", os.path.exists(base))

if os.path.exists(base):
    for root, dirs, files in os.walk(base):
        level = root.replace(base, "").count(os.sep)
        if level > 3:
            dirs[:] = []
            continue

        print("\n" + "  " * level + os.path.basename(root) + "/")

        for file in files:
            path = os.path.join(root, file)
            size = os.path.getsize(path) / (1024 * 1024)
            print("  " * (level + 1) + f"{file} ({size:.2f} MB)")

Training folder exists: False


In [83]:
# 2} Check the GPU and libraries
import torch
import transformers
import peft
import pandas as pd

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Pandas:", pd.__version__)

GPU available: True
GPU: Tesla T4
VRAM: 14.56 GB
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
PEFT: 0.21.2
Pandas: 2.3.3


In [84]:
# 3}Find the test dataset
import os

for base in ["/kaggle/working", "/kaggle/input"]:
    print(f"\n{'='*50}\nSEARCHING: {base}\n{'='*50}")

    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d != ".virtual_documents"]

        for file in files:
            if file.lower().endswith((
                ".csv", ".json", ".jsonl", ".parquet",
                ".xlsx", ".arrow", ".pkl"
            )):
                path = os.path.join(root, file)
                size = os.path.getsize(path) / (1024 * 1024)
                print(f"{path}  ({size:.2f} MB)")


SEARCHING: /kaggle/working
/kaggle/working/kaizenly-v2/checkpoint-1010/adapter_config.json  (0.00 MB)
/kaggle/working/kaizenly-v2/checkpoint-1010/trainer_state.json  (0.01 MB)
/kaggle/working/kaizenly-v2/checkpoint-1000/adapter_config.json  (0.00 MB)
/kaggle/working/kaizenly-v2/checkpoint-1000/trainer_state.json  (0.01 MB)
/kaggle/working/kaizenly-lora-adapter/tokenizer.json  (10.89 MB)
/kaggle/working/kaizenly-lora-adapter/adapter_config.json  (0.00 MB)
/kaggle/working/kaizenly-lora-adapter/tokenizer_config.json  (0.00 MB)

SEARCHING: /kaggle/input


In [ ]:
# Batch 1
import os
from pathlib import Path

print("=" * 60)
print("1. KAGGLE WORKING DIRECTORY")
print("=" * 60)
print(os.listdir("/kaggle/working"))

print("\n" + "=" * 60)
print("2. HUGGING FACE DATASET CACHE")
print("=" * 60)

cache = Path("/root/.cache/huggingface/datasets")
print("Cache exists:", cache.exists())

if cache.exists():
    for p in cache.iterdir():
        print(p)

print("\n" + "=" * 60)
print("3. OTHER CACHE DIRECTORIES")
print("=" * 60)

for path in ["/root/.cache/huggingface", "/kaggle/temp", "/tmp"]:
    print(f"\n{path} exists: {os.path.exists(path)}")
    if os.path.exists(path):
        print(os.listdir(path)[:30])

print("\n" + "=" * 60)
print("4. DATASETS CURRENTLY IN MEMORY")
print("=" * 60)

for name, value in list(globals().items()):
    try:
        if hasattr(value, "column_names") and hasattr(value, "num_rows"):
            print(name, value.num_rows, value.column_names)
    except Exception:
        pass

print("\n" + "=" * 60)
print("5. TRAINING-RELATED VARIABLES")
print("=" * 60)

for name, value in list(globals().items()):
    if any(word in name.lower() for word in [
        "train", "test", "valid", "dataset", "tokenized"
    ]):
        print(name, type(value).__name__)

print("\n" + "=" * 60)
print("CHECK COMPLETE")
print("=" * 60)

In [85]:
# Batch 2
import torch

print("=" * 60)
print("1. MODEL AND TOKENIZER")
print("=" * 60)

for name in [
    "model", "base_model", "peft_model",
    "trainer", "tokenizer"
]:
    if name in globals():
        obj = globals()[name]
        print(name, "->", type(obj).__name__)

print("\n" + "=" * 60)
print("2. TRAINING STATUS")
print("=" * 60)

if "trainer" in globals():
    print("Global step:", trainer.state.global_step)
    print("Training loss:", trainer.state.log_history[-1]
          if trainer.state.log_history else "No logs")
    print("Best checkpoint:", trainer.state.best_model_checkpoint)

print("\n" + "=" * 60)
print("3. TEST DATA SAMPLE")
print("=" * 60)

print("Test examples:", len(test_dataset))
print("Columns:", test_dataset.column_names)
print("\nFirst example:")
print(test_dataset[0]["messages"])

print("\n" + "=" * 60)
print("4. GPU MEMORY")
print("=" * 60)

if torch.cuda.is_available():
    print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
    print("Reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")
    print("Free:", round(
        (torch.cuda.get_device_properties(0).total_memory
         - torch.cuda.memory_reserved()) / 1024**3, 2
    ), "GB")

1. MODEL AND TOKENIZER
model -> PeftModelForCausalLM
trainer -> Trainer
tokenizer -> Qwen2Tokenizer

2. TRAINING STATUS
Global step: 1010
Training loss: {'train_runtime': 4583.0907, 'train_samples_per_second': 1.761, 'train_steps_per_second': 0.22, 'total_flos': 2.2302377608974336e+16, 'train_loss': 0.8489065812365844, 'epoch': 2.0, 'step': 1010}
Best checkpoint: ./kaizenly-v2/checkpoint-1010

3. TEST DATA SAMPLE
Test examples: 634
Columns: ['messages']

First example:
[{'content': "You are Kaizenly, an academic assignment feedback assistant.\nEvaluate the student's answer using the question, reference answer,\nand marking scheme.\n\nReturn valid JSON with these fields:\n- score: integer (0, 1, or 2)\n- performance: Incorrect, Partially Correct, or Correct\n- feedback: explanation of the evaluation\n\nUse the provided reference answer and rationale.\nDo not invent information or award unsupported marks.", 'role': 'system'}, {'content': "Question:\nc. Assume the OS is using a lazy alloc

In [87]:
# Batch 3
import torch

print("=" * 55)
print("1. MODEL CONFIGURATION")
print("=" * 55)

print("Model class:", type(model).__name__)
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

print("\n" + "=" * 55)
print("2. PEFT ADAPTER CONFIGURATION")
print("=" * 55)

print("Active adapter:", model.active_adapter)

for name, config in model.peft_config.items():
    print("Adapter:", name)
    print("Base model:", config.base_model_name_or_path)
    print("LoRA rank:", config.r)
    print("LoRA alpha:", config.lora_alpha)

print("\n" + "=" * 55)
print("3. TRAINABLE PARAMETERS")
print("=" * 55)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("Trainable:", f"{trainable:,}")
print("Total:", f"{total:,}")
print("Trainable percentage:", round(100 * trainable / total, 4), "%")

print("\n" + "=" * 55)
print("4. TRAINER HISTORY")
print("=" * 55)

print("Global step:", trainer.state.global_step)
print("Log history:", trainer.state.log_history[-5:])

print("\n" + "=" * 55)
print("5. ADAPTER FILES")
print("=" * 55)

import os

adapter_path = "/kaggle/working/kaizenly-lora-adapter"

for file in os.listdir(adapter_path):
    path = os.path.join(adapter_path, file)
    print(file, round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

print("\nVERIFICATION COMPLETE")

1. MODEL CONFIGURATION
Model class: PeftModelForCausalLM
Model device: cuda:0
Model dtype: torch.float32

2. PEFT ADAPTER CONFIGURATION
Active adapter: default
Adapter: default
Base model: Qwen/Qwen2.5-1.5B-Instruct
LoRA rank: 8
LoRA alpha: 16

3. TRAINABLE PARAMETERS
Trainable: 9,232,384
Total: 897,848,832
Trainable percentage: 1.0283 %

4. TRAINER HISTORY
Global step: 1010
Log history: [{'loss': 0.778376579284668, 'grad_norm': 1.282429814338684, 'learning_rate': 1.6145833333333335e-06, 'epoch': 1.9415262636273538, 'step': 980}, {'loss': 0.785080099105835, 'grad_norm': 1.615112543106079, 'learning_rate': 5.729166666666667e-07, 'epoch': 1.9811694747274529, 'step': 1000}, {'eval_loss': 0.8828144073486328, 'eval_runtime': 101.2647, 'eval_samples_per_second': 7.999, 'eval_steps_per_second': 3.999, 'epoch': 1.9811694747274529, 'step': 1000}, {'eval_loss': 0.8827704787254333, 'eval_runtime': 101.2189, 'eval_samples_per_second': 8.002, 'eval_steps_per_second': 4.001, 'epoch': 2.0, 'step': 10

In [88]:
# Batch 4
import torch
import json

model.eval()

def generate_feedback(example):
    messages = example["messages"]

    # Use system + user messages as input
    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    expected = messages[-1]["content"]

    return response, expected


print("=" * 65)
print("KAIZENLY — PILOT EVALUATION")
print("=" * 65)

pilot_results = []

for i in range(3):
    prediction, expected = generate_feedback(test_dataset[i])

    print(f"\n{'='*65}")
    print(f"EXAMPLE {i+1}")
    print("=" * 65)

    print("\nEXPECTED OUTPUT:")
    print(expected)

    print("\nMODEL PREDICTION:")
    print(prediction)

    try:
        parsed = json.loads(prediction)
        print("\nJSON VALID: YES")
        print("Predicted score:", parsed.get("score"))
        print("Performance:", parsed.get("performance"))
    except Exception:
        print("\nJSON VALID: NO")

    pilot_results.append({
        "prediction": prediction,
        "expected": expected
    })

print("\nPILOT COMPLETE")

KAIZENLY — PILOT EVALUATION

EXAMPLE 1

EXPECTED OUTPUT:
{"score": 0, "performance": "Incorrect", "feedback": "The student response is incorrect. The OS page fault handler does not execute the next instruction after fixing the mapping. It creates the mappings, returns from the handler, and then re-executes the faulting instruction."}

MODEL PREDICTION:
{"score": 0, "performance": "Incorrect", "feedback": "Your understanding of how the operating system handles page faults is incorrect. The page fault handler does not increment the program counter (PC). Instead, it creates new mappings and returns from the handler before re-executing the faulty instruction."}

JSON VALID: YES
Predicted score: 0
Performance: Incorrect

EXAMPLE 2

EXPECTED OUTPUT:
{"score": 2, "performance": "Correct", "feedback": "You appear to have provided a correct reasoning for the 'Correct Response' feedback label. While your feedback is concise and guides the learner, it is essential to avoid using subjective phrase

In [90]:
# Batch 5
import json
import re

def clean_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found")

    return json.loads(text[start:end + 1])


correct_scores = 0
valid_json = 0
valid_labels = 0

for i, result in enumerate(pilot_results):
    try:
        predicted = clean_json(result["prediction"])
        expected = json.loads(result["expected"])

        score_ok = predicted["score"] == expected["score"]
        label_ok = predicted.get("performance") in [
            "Incorrect", "Partially Correct", "Correct"
        ]

        valid_json += 1
        valid_labels += int(label_ok)
        correct_scores += int(score_ok)

        print(f"\nExample {i+1}")
        print("Predicted:", predicted["score"])
        print("Expected:", expected["score"])
        print("Score match:", score_ok)
        print("Valid label:", label_ok)
        print("Feedback:", bool(predicted.get("feedback")))

    except Exception as e:
        print(f"Example {i+1} error:", e)

print("\n========== PILOT SUMMARY ==========")
print("Valid JSON:", valid_json, "/ 3")
print("Valid labels:", valid_labels, "/ 3")
print("Correct scores:", correct_scores, "/ 3")
print("Accuracy:", round(correct_scores / 3 * 100, 2), "%")


Example 1
Predicted: 0
Expected: 0
Score match: True
Valid label: True
Feedback: True

Example 2
Predicted: 2
Expected: 2
Score match: True
Valid label: True
Feedback: True

Example 3
Predicted: 2
Expected: 2
Score match: True
Valid label: True
Feedback: True

========== PILOT SUMMARY ==========
Valid JSON: 3 / 3
Valid labels: 3 / 3
Correct scores: 3 / 3
Accuracy: 100.0 %


In [91]:
# Batch 6
import torch
import json
import re
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

model.eval()

def parse_model_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found")

    return json.loads(text[start:end + 1])


results = []
valid_predictions = []
expected_scores = []
predicted_scores = []

print("Starting evaluation on", len(test_dataset), "examples")

for i in tqdm(range(len(test_dataset))):

    messages = test_dataset[i]["messages"]

    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    expected = json.loads(messages[-1]["content"])

    record = {
        "index": i,
        "expected_score": expected.get("score"),
        "predicted_score": None,
        "valid_json": False,
        "valid_label": False,
        "feedback_present": False,
        "prediction": response
    }

    try:
        parsed = parse_model_json(response)

        score = parsed.get("score")
        label = parsed.get("performance")
        feedback = parsed.get("feedback")

        valid_score = (
            isinstance(score, int)
            and not isinstance(score, bool)
            and score in [0, 1, 2]
        )

        valid_label = label in [
            "Incorrect",
            "Partially Correct",
            "Correct"
        ]

        valid_feedback = (
            isinstance(feedback, str)
            and len(feedback.strip()) > 0
        )

        record["valid_json"] = True
        record["valid_label"] = valid_label
        record["feedback_present"] = valid_feedback

        if valid_score:
            record["predicted_score"] = score
            valid_predictions.append((expected["score"], score))
            expected_scores.append(expected["score"])
            predicted_scores.append(score)

    except Exception:
        pass

    results.append(record)


# Save detailed predictions
results_df = pd.DataFrame(results)

output_path = "/kaggle/working/kaizenly_finetuned_test_predictions.csv"
results_df.to_csv(output_path, index=False)

# Calculate metrics
total = len(results_df)
valid_json_count = int(results_df["valid_json"].sum())
valid_label_count = int(results_df["valid_label"].sum())
feedback_count = int(results_df["feedback_present"].sum())

correct = sum(
    actual == predicted
    for actual, predicted in valid_predictions
)

strict_accuracy = correct / total
valid_accuracy = (
    accuracy_score(expected_scores, predicted_scores)
    if expected_scores else 0
)

print("\n" + "=" * 60)
print("KAIZENLY — FULL TEST EVALUATION")
print("=" * 60)

print("Total examples:", total)
print("Valid JSON:", valid_json_count, "/", total)
print("Valid labels:", valid_label_count, "/", total)
print("Feedback present:", feedback_count, "/", total)
print("Valid score predictions:", len(valid_predictions), "/", total)

print("\nStrict accuracy (invalid outputs count as wrong):",
      round(strict_accuracy * 100, 2), "%")

print("Accuracy on valid score predictions:",
      round(valid_accuracy * 100, 2), "%")

if expected_scores:
    precision, recall, f1, _ = precision_recall_fscore_support(
        expected_scores,
        predicted_scores,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0
    )

    print("\nMacro precision:", round(precision, 4))
    print("Macro recall:", round(recall, 4))
    print("Macro F1:", round(f1, 4))

    print("\nClassification report:")
    print(classification_report(
        expected_scores,
        predicted_scores,
        labels=[0, 1, 2],
        target_names=[
            "Incorrect (0)",
            "Partially Correct (1)",
            "Correct (2)"
        ],
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(
        expected_scores,
        predicted_scores,
        labels=[0, 1, 2]
    ))

print("\nPredictions saved to:", output_path)
print("=" * 60)

Starting evaluation on 634 examples


  0%|          | 0/634 [00:00<?, ?it/s]


KAIZENLY — FULL TEST EVALUATION
Total examples: 634
Valid JSON: 634 / 634
Valid labels: 634 / 634
Feedback present: 634 / 634
Valid score predictions: 634 / 634

Strict accuracy (invalid outputs count as wrong): 60.57 %
Accuracy on valid score predictions: 60.57 %

Macro precision: 0.5643
Macro recall: 0.5949
Macro F1: 0.5415

Classification report:
                       precision    recall  f1-score   support

        Incorrect (0)       0.46      0.82      0.59       137
Partially Correct (1)       0.48      0.19      0.28       192
          Correct (2)       0.75      0.77      0.76       305

             accuracy                           0.61       634
            macro avg       0.56      0.59      0.54       634
         weighted avg       0.61      0.61      0.58       634

Confusion matrix:
[[113   7  17]
 [ 96  37  59]
 [ 38  33 234]]

Predictions saved to: /kaggle/working/kaizenly_finetuned_test_predictions.csv


In [ ]:
import os
import shutil

source = "/kaggle/working/kaizenly_finetuned_test_predictions.csv"
destination = "/kaggle/working/kaizenly_test_results.zip"

assert os.path.exists(source), "CSV file not found!"

shutil.make_archive(
    "/kaggle/working/kaizenly_test_results",
    "zip",
    root_dir="/kaggle/working",
    base_dir="kaizenly_finetuned_test_predictions.csv"
)

print("ZIP created:", os.path.exists(destination))
print("ZIP size:", round(os.path.getsize(destination) / 1024, 2), "KB")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
# 1> Clean GPU memory and imports
import os, gc, torch, transformers, peft
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig, prepare_model_for_kbit_training,
    get_peft_model
)

gc.collect()
torch.cuda.empty_cache()

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/kaggle/working/kaizenly-v2"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

In [ ]:
# 2>Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer ready:", len(tokenizer))

In [ ]:
# 3>Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

print("Base model loaded on GPU 0!")

In [2]:
!pip install -q --no-cache-dir "bitsandbytes>=0.46.1"

In [3]:
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.50.2
